In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from tqdm.auto import tqdm

In [2]:
jsonl_path = Path("../results/ncbi_bacteria_tree.jsonl")
with jsonl_path.open("r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(rows)} rows from {jsonl_path}")

Loaded 101283 rows from ../results/ncbi_bacteria_tree.jsonl


In [3]:
complete_count = len([row for row in rows if str(row.get("sub_dirs", []))])
complete_count

101283

In [4]:
_df = []
ROW_TEMPLATE = {
    "bacteria_name": "",
    "source_url": "",
    "is_dir": False,
    "folder_name": "",
}

status_counts = {}

for row in tqdm(rows):
    if row.get("status", "") != "done":
        continue
    for dir in row.get("sub_dirs", []):
        _row = ROW_TEMPLATE.copy()
        _row["bacteria_name"] = row.get("bacteria_name", "")
        _row["source_url"] = dir.get("source_url", "")
        _row["is_dir"] = dir.get("is_dir", False)
        _row["folder_name"] = dir.get("bacteria_name", "")
        _df.append(_row.copy())

  0%|          | 0/101283 [00:00<?, ?it/s]

In [5]:
df = pd.DataFrame(_df)

In [6]:
latest_df = df[df.folder_name == "latest_assembly_versions/"]

In [7]:
latest_df.shape[0], latest_df.bacteria_name.nunique()

(100349, 100349)

In [10]:
# save the latest_df to a jsonl file
output_path = Path("../results/latest_assembly_versions.jsonl")
with output_path.open("w", encoding="utf-8") as f:
    for _, row in latest_df.iterrows():
        json.dump(row.to_dict(), f)
        f.write("\n")